# Flight Price Prediction - Models

Steps:
1. features (basic ones + target encoding + id order features)
2. compare models
3. tuning
4. mixture model for expensive tickets, stacking and blending
5. final submission

Takes a while to run (mostly random forest and optuna).

In [ ]:
import numpy as np
import pandas as pd
import time
import optuna
from scipy.optimize import minimize, nnls
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor, XGBClassifier

SEED = 42

In [ ]:
path = "/kaggle/input/competitions/flight-price-prediction/"
train = pd.read_csv(path + "train.csv")
test = pd.read_csv(path + "test.csv")
sample_sub = pd.read_csv(path + "submission.csv")

n_tr = len(train)
y_raw = train["price"].values.astype(float)
y = np.log1p(y_raw)   # log price
print(train.shape, test.shape)

## Basic features

In [ ]:
df = pd.concat([train.drop(columns="price"), test], ignore_index=True)

stops_map = {"zero": 0, "one": 1, "two_or_more": 2}
time_map = {"Early_Morning": 0, "Morning": 1, "Afternoon": 2, "Evening": 3, "Night": 4, "Late_Night": 5}

df["stops_n"] = df["stops"].map(stops_map)
df["dep_n"] = df["departure_time"].map(time_map)
df["arr_n"] = df["arrival_time"].map(time_map)
df["is_business"] = (df["flight_class"] == "Business").astype(int)
df["route"] = df["source_city"] + "_" + df["destination_city"]
df["flight_prefix"] = df["flight"].str.split().str[0]
df["flight_num"] = pd.to_numeric(df["flight"].str.split().str[-1], errors="coerce").fillna(-1)
df["overnight"] = (df["arr_n"] < df["dep_n"]).astype(int)
df["dur_per_leg"] = df["duration"] / (df["stops_n"] + 1)
df["log_duration"] = np.log1p(df["duration"])
df["days_left_log"] = np.log1p(df["days_left"])
df["last_week"] = (df["days_left"] <= 7).astype(int)
df["dl_bucket"] = pd.cut(df["days_left"], [0, 3, 7, 14, 21, 30, 50], labels=False)

# combo keys (used for target encoding)
df["flight_route_class"] = df["flight"] + "|" + df["route"] + "|" + df["flight_class"]
df["flight_class_key"] = df["flight"] + "|" + df["flight_class"]
df["route_class"] = df["route"] + "|" + df["flight_class"]
df["airline_class"] = df["airline"] + "|" + df["flight_class"]
df["route_stops_class"] = df["route"] + "|" + df["stops"] + "|" + df["flight_class"]
df["frc_dl"] = df["flight_route_class"] + "|" + df["dl_bucket"].astype(str)
df["route_class_dl"] = df["route_class"] + "|" + df["dl_bucket"].astype(str)
df["airline_route_class"] = df["airline"] + "|" + df["route_class"]

# frequency + label encoding
for c in ["flight", "route", "airline", "flight_route_class"]:
    df[c + "_freq"] = df[c].map(df[c].value_counts())

cat_cols = ["airline", "source_city", "departure_time", "stops", "arrival_time",
            "destination_city", "flight_class", "route", "flight", "flight_prefix"]
for c in cat_cols:
    df[c + "_le"] = df[c].astype("category").cat.codes

base_feats = ["duration", "log_duration", "days_left", "days_left_log", "stops_n", "dep_n", "arr_n",
              "is_business", "flight_num", "overnight", "dur_per_leg", "last_week", "dl_bucket",
              "flight_freq", "route_freq", "airline_freq", "flight_route_class_freq"]
base_feats += [c + "_le" for c in cat_cols]
print(len(base_feats), "base features")

## id order features
From EDA: id follows the scrape order and prices are sorted inside blocks of same (days_left, route, class).
These features only use train prices, never the row's own price.

In [ ]:
N = len(df)
n_te = len(test)
ids = df["id"].values

# known log price for train rows (zero prices treated as unknown), nan for test
lp_row = np.r_[np.where(y_raw > 0, y, np.nan), np.full(n_te, np.nan)]
order = np.argsort(ids, kind="stable")
id_feats = ["id"]

# nearest known neighbours (by id)
pos = np.empty(N, int)
pos[order] = np.arange(N)
P_known = np.sort(pos[~np.isnan(lp_row)])
jl = np.searchsorted(P_known, pos, "left")
jr = np.searchsorted(P_known, pos, "right")

for k in [1, 2]:
    for side, j in [("p", jl - k), ("n", jr + k - 1)]:
        ok = (j >= 0) & (j < len(P_known))
        nb = order[P_known[np.clip(j, 0, len(P_known) - 1)]]
        f = side + str(k)
        df[f + "_lp"] = np.where(ok, lp_row[nb], np.nan)
        df[f + "_gap"] = np.where(ok, np.abs(ids[nb] - ids), np.nan)
        id_feats += [f + "_lp", f + "_gap"]
        for c in ["flight", "route", "flight_class", "days_left"]:
            df[f + "_same_" + c] = np.where(ok, (df[c].values[nb] == df[c].values).astype(float), np.nan)
            id_feats.append(f + "_same_" + c)

In [ ]:
# block features
S = pd.DataFrame({"row": order, "id": ids[order].astype(float), "lp": lp_row[order]})
for c in ["days_left", "route", "flight_class"]:
    S[c] = df[c].values[order]
key = S["days_left"].astype(str) + "|" + S["route"] + "|" + S["flight_class"]
S["run"] = (key != key.shift()).cumsum()
S["id_known"] = np.where(S["lp"].notna(), S["id"], np.nan)

g = S.groupby("run")
S["rp_lp"] = g["lp"].transform(lambda x: x.shift(1).ffill())          # previous known price in block
S["rp_id"] = g["id_known"].transform(lambda x: x.shift(1).ffill())
S["rn_lp"] = g["lp"].transform(lambda x: x.shift(-1).bfill())         # next known price in block
S["rn_id"] = g["id_known"].transform(lambda x: x.shift(-1).bfill())
S["rp_gap"] = S["id"] - S["rp_id"]
S["rn_gap"] = S["rn_id"] - S["id"]
S["bound_width"] = S["rn_lp"] - S["rp_lp"]

# linear interpolation between prev and next price
pp = np.expm1(S["rp_lp"])
pn = np.expm1(S["rn_lp"])
frac = (S["id"] - S["rp_id"]) / (S["rn_id"] - S["rp_id"])
both = S["rp_lp"].notna() & S["rn_lp"].notna()
S["interp_price"] = np.where(both, pp + (pn - pp) * frac, np.where(S["rp_lp"].notna(), pp, pn))
S["interp_lp"] = np.log1p(S["interp_price"])

known = S["lp"].notna().astype(int)
S["run_known_before"] = known.groupby(S["run"]).cumsum() - known
S["run_others"] = known.groupby(S["run"]).transform("sum") - known
S["run_rank_pct"] = S["run_known_before"] / S["run_others"].replace(0, np.nan)
S["run_mean_lp"] = (S["lp"].fillna(0).groupby(S["run"]).transform("sum") - S["lp"].fillna(0)) / S["run_others"].replace(0, np.nan)
S["run_len"] = g["id"].transform("size")
S["run_pos"] = g.cumcount()
S["run_pct"] = S["run_pos"] / np.maximum(S["run_len"] - 1, 1)

run_feats = ["rp_lp", "rn_lp", "rp_gap", "rn_gap", "bound_width", "interp_price", "interp_lp",
             "run_known_before", "run_others", "run_rank_pct", "run_mean_lp", "run_len", "run_pos", "run_pct"]
for c in run_feats:
    arr = np.empty(N)
    arr[S["row"].values] = S[c].values
    df[c] = arr
id_feats += run_feats
df[id_feats] = df[id_feats].fillna(-1)

D_tr = df.iloc[:n_tr].reset_index(drop=True)
D_te = df.iloc[n_tr:].reset_index(drop=True)
print(len(id_feats), "id features")

# how good is interpolation alone?
ip = D_tr["interp_price"].values
m = ip >= 0
print("interpolation only R2:", round(r2_score(y_raw[m], ip[m]), 4), "| rows covered:", round(m.mean() * 100, 1), "%")

## Target encoding
Mean (log) price per group. Done out-of-fold with an inner 5-fold split so the row's own price is never used.

In [ ]:
te_keys = ["flight", "flight_class_key", "flight_route_class", "route_class", "airline_class",
           "route_stops_class", "frc_dl", "route_class_dl", "airline_route_class"]
rich_keys = ["flight_route_class", "flight_class_key", "frc_dl"]   # these also get raw mean + std


def te_stats(tr_k, tr_yl, tr_yr, ap_k, m=5):
    s = pd.DataFrame({"k": tr_k, "yl": tr_yl, "yr": tr_yr}).groupby("k").agg(
        n=("yl", "size"), sl=("yl", "sum"), sr=("yr", "sum"), sd=("yl", "std"))
    gl, gr = tr_yl.mean(), tr_yr.mean()
    ml = (s.sl + m * gl) / (s.n + m)   # smoothed mean
    mr = (s.sr + m * gr) / (s.n + m)
    k = pd.Series(ap_k)
    return k.map(ml).fillna(gl).values, k.map(mr).fillna(gr).values, k.map(s.sd).fillna(-1).values


def add_te(d_tr, yl, yr, d_ap):
    tr_out, ap_out = {}, {}
    inner = KFold(5, shuffle=True, random_state=0)
    for k in te_keys:
        kv = d_tr[k].values
        names = ["te_" + k]
        if k in rich_keys:
            names += ["teraw_" + k, "testd_" + k]
        oof = np.zeros((len(kv), 3))
        for a, b in inner.split(kv):
            oof[b] = np.column_stack(te_stats(kv[a], yl[a], yr[a], kv[b]))
        full = np.column_stack(te_stats(kv, yl, yr, d_ap[k].values))
        for j, nm in enumerate(names):
            tr_out[nm] = oof[:, j]
            ap_out[nm] = full[:, j]
    return pd.DataFrame(tr_out), pd.DataFrame(ap_out)


def build(tr_idx, ap_df, feats):
    d_tr = D_tr.iloc[tr_idx].reset_index(drop=True)
    te_tr, te_ap = add_te(d_tr, y[tr_idx], y_raw[tr_idx], ap_df.reset_index(drop=True))
    Xtr = pd.concat([d_tr[feats], te_tr], axis=1)
    Xap = pd.concat([ap_df[feats].reset_index(drop=True), te_ap], axis=1)
    return Xtr, Xap

## CV setup
Same 5 folds for all models. Zero price rows are removed from training but kept in validation.
Fold data is built once per feature set and reused.

In [ ]:
keep = y_raw > 0
kf = KFold(5, shuffle=True, random_state=SEED)
folds = list(kf.split(D_tr))


def make_fold_data(feats):
    data = []
    for tr_idx, va_idx in folds:
        tr_idx = tr_idx[keep[tr_idx]]
        Xtr, Xva = build(tr_idx, D_tr.iloc[va_idx], feats)
        data.append((tr_idx, va_idx, Xtr, Xva))
    return data


def show(name, pred):
    m = y_raw < y_raw.max()   # trim = without the single biggest price
    print(f"{name:16s} R2 {r2_score(y_raw, pred):.4f} | RMSE {np.sqrt(mean_squared_error(y_raw, pred)):.2f} "
          f"| RMSE_trim {np.sqrt(mean_squared_error(y_raw[m], pred[m])):.2f} | MAE {mean_absolute_error(y_raw, pred):.2f}")
    return [name, r2_score(y_raw, pred), np.sqrt(mean_squared_error(y_raw, pred)),
            np.sqrt(mean_squared_error(y_raw[m], pred[m])), mean_absolute_error(y_raw, pred)]


def run_cv(fit_pred, data):
    # fit_pred(Xtr, y_raw_tr, Xva) -> price predictions
    oof = np.zeros(n_tr)
    for tr_idx, va_idx, Xtr, Xva in data:
        oof[va_idx] = fit_pred(Xtr, y_raw[tr_idx], Xva)
    return np.clip(oof, 0, None)


def log_model(model):
    # train on log price, return price
    return lambda Xtr, yr, Xva: np.expm1(model.fit(Xtr, np.log1p(yr)).predict(Xva))


def raw_model(model):
    return lambda Xtr, yr, Xva: model.fit(Xtr, yr).predict(Xva)


t = time.time()
data_base = make_fold_data(base_feats)
print("fold data ready", round(time.time() - t), "s")

## Model comparison (basic features, log target)

In [ ]:
models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge()),
    "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor()),
    "Decision Tree": DecisionTreeRegressor(random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=SEED),
}

base_results = []
for name, model in models.items():
    t = time.time()
    oof = run_cv(log_model(model), data_base)
    base_results.append(show(name, oof))
    print("   ", round(time.time() - t), "s")

## Tuning XGBoost with optuna
40 trials (or 25 min max), each scored on the 5 folds with early stopping.

In [ ]:
data_all = make_fold_data(base_feats + id_feats)   # basic + id features


def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 5, log=True),
    }
    oof = np.zeros(n_tr)
    trees = []
    for tr_idx, va_idx, Xtr, Xva in data_all:
        model = XGBRegressor(n_estimators=3000, early_stopping_rounds=100, random_state=SEED, **params)
        model.fit(Xtr, np.log1p(y_raw[tr_idx]), eval_set=[(Xva, np.log1p(y_raw[va_idx]))], verbose=False)
        oof[va_idx] = np.expm1(model.predict(Xva))
        trees.append(model.best_iteration + 1)
    trial.set_user_attr("trees", int(np.mean(trees)))
    return r2_score(y_raw, oof)


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=40, timeout=1500)
print(study.best_params)
print("trees:", study.best_trial.user_attrs["trees"], "| R2:", round(study.best_value, 4))

In [ ]:
# rounded version of what optuna found
xgb_params = dict(n_estimators=1400, learning_rate=0.025, max_depth=9, min_child_weight=2,
                  subsample=0.8, colsample_bytree=0.45, reg_lambda=9, reg_alpha=0.2, random_state=SEED)

# does the id order stuff help?
show("XGB basic", run_cv(log_model(XGBRegressor(**xgb_params)), data_base))
xgb_oof = run_cv(log_model(XGBRegressor(**xgb_params)), data_all)
show("XGB basic+id", xgb_oof)

m = XGBRegressor(**xgb_params).fit(data_all[0][2], np.log1p(y_raw[data_all[0][0]]))
imp = pd.Series(m.get_booster().get_score(importance_type="gain"))
print((imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(1))

## Random forest grid search

In [ ]:
for mf in [0.3, 0.5, 0.7]:
    for leaf in [1, 3]:
        rf = RandomForestRegressor(n_estimators=200, max_features=mf, min_samples_leaf=leaf, random_state=SEED, n_jobs=-1)
        oof = run_cv(log_model(rf), data_all)
        print("max_features", mf, "leaf", leaf, "R2", round(r2_score(y_raw, oof), 4))

## Mixture model for expensive tickets
Normal tickets and very expensive ones (> T) behave differently, so:
- classifier gives p = chance price > T
- xgb trained on normal prices (log)
- random forest trained on expensive prices
- prediction = (1 - p) * normal + p * expensive

In [ ]:
def mixture(Xtr, yr, Xap, T=1000):
    hi = yr > T
    gate = XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=6, subsample=0.8,
                         colsample_bytree=0.8, random_state=SEED)
    p = gate.fit(Xtr, hi.astype(int)).predict_proba(Xap)[:, 1]
    lo = np.expm1(XGBRegressor(**xgb_params).fit(Xtr[~hi], np.log1p(yr[~hi])).predict(Xap))
    hp = RandomForestRegressor(n_estimators=500, max_features=0.5, min_samples_leaf=2,
                               random_state=SEED, n_jobs=-1).fit(Xtr[hi], yr[hi]).predict(Xap)
    return np.clip((1 - p) * lo + p * hp, 0, None)

In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

oofs = {}
oofs["XGB_log"] = xgb_oof
oofs["XGB_raw"] = run_cv(raw_model(XGBRegressor(**xgb_params)), data_all)
oofs["RF_log"] = run_cv(log_model(rf), data_all)
oofs["RF_raw"] = run_cv(raw_model(rf), data_all)
oofs["MIX_T600"] = run_cv(lambda a, b, c: mixture(a, b, c, T=600), data_all)
oofs["MIX_T1000"] = run_cv(lambda a, b, c: mixture(a, b, c, T=1000), data_all)

for k, v in oofs.items():
    show(k, v)

## Stacking / blending

In [ ]:
names = list(oofs)
M = np.column_stack([oofs[k] for k in names])

# stacking: ridge on log predictions
stack = np.expm1(cross_val_predict(Ridge(), np.log1p(M), y, cv=kf))
oofs["STACK_ridgelog"] = np.clip(stack, 0, None)
print("ridge weights:", {k: round(float(w), 3) for k, w in zip(names, Ridge().fit(np.log1p(M), y).coef_)})

# blend: positive weights that add up to 1
loss = lambda w: mean_squared_error(y_raw, M @ w)
res = minimize(loss, np.ones(len(names)) / len(names), bounds=[(0, 1)] * len(names),
               constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
oofs["BLEND"] = M @ res.x
print("blend weights:", {k: round(float(w), 3) for k, w in zip(names, res.x)})

# non negative linear stack (just to check)
w, _ = nnls(M, y_raw)
print("nnls stack R2:", round(r2_score(y_raw, M @ w), 4), {k: round(float(x), 3) for k, x in zip(names, w)})

# simple average
oofs["FINAL_avg"] = M.mean(axis=1)

table = pd.DataFrame([show(k, v) for k, v in oofs.items()], columns=["model", "R2", "RMSE", "RMSE_trim", "MAE"])
print(table.sort_values("R2", ascending=False).round(4))

MIX_T1000 was the best on CV and on the leaderboard, so that's the final one.
## Final model + submission

In [ ]:
full_idx = np.where(keep)[0]
X_full, X_test = build(full_idx, D_te, base_feats + id_feats)
pred = mixture(X_full, y_raw[full_idx], X_test, T=1000)

sub = pd.DataFrame({"id": test["id"].values, "price": pred})
print(sub.shape, (sub["id"].values == sample_sub["id"].values).all())
print(sub["price"].describe().round(1))
sub.to_csv("submission.csv", index=False)